# Create a file simple_agent_for_calulator_weather.py (No Langchain)
Install
```
pip install openai requests

In [3]:
# Check version
import sys, openai, requests

print("Python:", sys.version)
print("OpenAI:", openai.__version__)
print("Requests:", requests.__version__)

Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]
OpenAI: 2.29.0
Requests: 2.32.5


### Try
```
input1: What is 1 + 2 * 4
input2: What is the current weather condition in Moscow ?
input3: I am trying to calculate this math expression. What is one plus two times four. What is one plus two times four
input4: I am living in Dallas, Texas. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?

In [ ]:
# File: simple_agent_for_calulator_weather.py

from openai import OpenAI
import requests
import re

API_KEY = "key_here"
client = OpenAI(api_key=API_KEY)

# -------------------------
# TOOL 1: Calculator
# -------------------------
def calculator(expression):
    print("...Running calculator tool")
    try:
        return str(eval(expression))
    except:
        return "Error in calculation"


# -------------------------
# TOOL 2: Weather (Open-Meteo API)
# -------------------------
def get_weather(city):
    print("...Running get_weather tool")
    try:
        # Step 1: Convert city → lat/lon
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
        geo_res = requests.get(geo_url).json()

        if "results" not in geo_res:
            return "City not found"

        lat = geo_res["results"][0]["latitude"]
        lon = geo_res["results"][0]["longitude"]

        # Step 2: Get weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_res = requests.get(weather_url).json()

        temp = weather_res["current_weather"]["temperature"]
        wind = weather_res["current_weather"]["windspeed"]

        return f"Temperature: {temp}°C, Wind Speed: {wind} km/h"

    except:
        return "Error fetching weather"


# -------------------------
# AGENT FUNCTION
# -------------------------
def ai_agent(user_input):

    system_prompt = """
    You are an AI agent.

    You can:
    1. Answer normally
    2. Use a calculator
    3. Get weather of a city

    Rules:
    - For math → respond EXACTLY:
      ACTION: CALCULATE <expression>

    - For weather → respond EXACTLY:
      ACTION: WEATHER <city>

    - Otherwise → respond normally
    """

    # Step 1: LLM decides action
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    output = response.choices[0].message.content
    print("LLM Decision:", output)

    # -------------------------
    # TOOL: Calculator
    # -------------------------
    if "ACTION: CALCULATE" in output:
        expression = re.findall(r'CALCULATE (.*)', output)[0]

        result = calculator(expression)
        print("Tool Result:", result)

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Give final answer in a professional tone. \
                                               Rule: Do not solve the expression. Do not verify the expression. \
                                               Do not tell me if the answer is correct or incorrect. \
                                               In the output, state ONLY the expression and its result in plain english, not numeric.\
                                               "},
                {"role": "user", "content": f"For Expression {expression}, the Result is {result}"}
            ]
        )

        return final_response.choices[0].message.content

    # -------------------------
    # TOOL: Weather
    # -------------------------
    elif "ACTION: WEATHER" in output:
        city = re.findall(r'WEATHER (.*)', output)[0]

        result = get_weather(city)
        print("Tool Result:", result)

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Explain weather in a friendly way."},
                {"role": "user", "content": f"City: {city}, Weather: {result}"}
            ]
        )

        return final_response.choices[0].message.content

    else:
        return output


# -------------------------
# main function that RUNS AGENT
# -------------------------
def main():
    while True:
        user_input = input("\nAsk something: ")
        answer = ai_agent(user_input)
        print("Final Answer:", answer)


if __name__ == "__main__":
    main()


Ask something:  What is 1 + 2 * 4


LLM Decision: ACTION: CALCULATE 1 + 2 * 4
...Running calculator tool
Tool Result: 9
Final Answer: The expression "1 + 2 * 4" results in 9.



Ask something:  What is the current weather condition in Moscow ?


LLM Decision: ACTION: WEATHER Moscow
...Running get_weather tool
Tool Result: Temperature: 5.0°C, Wind Speed: 3.8 km/h
Final Answer: Hello there! In Moscow right now, it’s a crisp 5 degrees Celsius, which is just cool enough to make you feel refreshed! It's a great excuse to grab your favorite cozy sweater or jacket before heading outside. The wind is gentle at 3.8 km/h, which means it shouldn’t be too breezy as you stroll around. Overall, it sounds like a lovely day to enjoy some fresh air. Maybe a nice warm drink could be the perfect complement to the cool weather! Stay warm and enjoy your day! 🌬️☕️🍂



Ask something:  I am trying to calculate this math expression. What is one plus two times four


LLM Decision: ACTION: CALCULATE 1 + 2 * 4
...Running calculator tool
Tool Result: 9
Final Answer: The expression 1 plus 2 times 4 results in 9.



Ask something:  I am living in Dallas, Texas. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?


LLM Decision: ACTION: WEATHER Dallas, Texas
...Running get_weather tool
Tool Result: City not found
Final Answer: Oh no! It looks like we weren’t able to find the weather for Dallas, Texas right now. But don’t worry! In general, Dallas can have some exciting weather. 

Typically, the city enjoys warm summers, full of bright sunshine and temperatures that can soar into the 90s °F (and sometimes even higher). Spring and fall bring mild temperatures and beautiful blooms, making those seasons a favorite for many. Winters are usually mild, with temperatures that can dip into the 30s °F at night but can also have some sunny days.

If you’re curious about what’s happening today or in the upcoming days, checking a weather app or website can give you the latest scoop. Just remember to stay hydrated and enjoy whatever the weather brings! 🌤️


# A Trial run
Ask something:  What is 1 + 2 * 4
LLM Decision: ACTION: CALCULATE 1 + 2 * 4
...Running calculator tool
Tool Result: 9
Final Answer: The expression "1 + 2 * 4" results in 9.

Ask something:  What is the current weather condition in Moscow ?
LLM Decision: ACTION: WEATHER Moscow
...Running get_weather tool
Tool Result: Temperature: 5.0°C, Wind Speed: 3.8 km/h
Final Answer: Hello there! In Moscow right now, it’s a crisp 5 degrees Celsius, which is just cool enough to make you feel refreshed! It's a great excuse to grab your favorite cozy sweater or jacket before heading outside. The wind is gentle at 3.8 km/h, which means it shouldn’t be too breezy as you stroll around. Overall, it sounds like a lovely day to enjoy some fresh air. Maybe a nice warm drink could be the perfect complement to the cool weather! Stay warm and enjoy your day! 🌬️☕️🍂

Ask something:  I am trying to calculate this math expression. What is one plus two times four
LLM Decision: ACTION: CALCULATE 1 + 2 * 4
...Running calculator tool
Tool Result: 9
Final Answer: The expression 1 plus 2 times 4 results in 9.

Ask something:  I am living in Dallas, Texas. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?
LLM Decision: ACTION: WEATHER Dallas, Texas
...Running get_weather tool
Tool Result: City not found
Final Answer: Oh no! It looks like we weren’t able to find the weather for Dallas, Texas right now. But don’t worry! In general, Dallas can have some exciting weather. 

Typically, the city enjoys warm summers, full of bright sunshine and temperatures that can soar into the 90s °F (and sometimes even higher). Spring and fall bring mild temperatures and beautiful blooms, making those seasons a favorite for many. Winters are usually mild, with temperatures that can dip into the 30s °F at night but can also have some sunny days.

If you’re curious about what’s happening today or in the upcoming days, checking a weather app or website can give you the latest scoop. Just remember to stay hydrated and enjoy whatever the weather brings! 🌤️